In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. The Neural Network Blueprint
class PolicyNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, num_actions):
        super(PolicyNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, num_actions)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x) 

# 2. The Car Environment
class CarEnv:
    def __init__(self, num_sensors=5, max_steps=50):
        assert num_sensors % 2 == 1
        
        self.num_sensors = num_sensors
        self.max_steps = max_steps
        
        self.min_speed = 0.001
        self.max_speed = 0.1
        
        self.reset()

    def reset(self):
        self.x = 0.5
        self.y = 0.5
        self.angle = 0.0
        self.speed = 0.01
        self.step_count = 0
        return self._get_state()

    def _get_sensor_readings(self):
        return [0.5] * self.num_sensors  # dummy for now

    def _get_state(self):
        return np.array(
            [self.x, self.y, self.angle, self.speed] + self._get_sensor_readings(),
            dtype=np.float32
        )

    def step(self, action):
        self.step_count += 1

        old_x, old_y = self.x, self.y

        # Action logic (minimal but correct mapping)
        if action == 0:  # left
            self.angle -= 0.01
            self.speed *= 0.9875
        elif action == 1:  # right
            self.angle += 0.01
            self.speed *= 0.9875
        elif action == 2:  # speed up
            self.speed *= 1.025
        elif action == 3:
            pass

        self.speed = max(self.min_speed, min(self.speed, self.max_speed))

        # Move
        self.x += self.speed * math.cos(self.angle)
        self.y += self.speed * math.sin(self.angle)

        # Clamp to bounds (simple crash handling)
        self.x = min(max(self.x, 0.0), 1.0)
        self.y = min(max(self.y, 0.0), 1.0)

        # Reward = distance moved
        dist = math.sqrt((self.x - old_x)**2 + (self.y - old_y)**2)

        done = self.step_count >= self.max_steps

        return self._get_state(), dist, done

def generate_trajectory(network, env):
    trajectory = []
    state = env.reset()
    done = False

    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        logits = network(state_tensor)

        probs = torch.softmax(logits, dim=0)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample().item()   # IMPORTANT

        next_state, reward, done = env.step(action)

        trajectory.append({
            'state': state,
            'action': action,
            'reward': reward
        })

        state = next_state

    return trajectory

# 4. The Trainer (Slightly improved to batch updates)
def train_reinforce(network, optimizer, trajectories):
    optimizer.zero_grad() # Clear old math
    total_loss = 0
    
    for trajectory in trajectories:
        total_reward = sum([step['reward'] for step in trajectory])
        
        for step in trajectory:
            state_tensor = torch.FloatTensor(step['state'])
            action_taken = step['action']
            
            action_scores = network(state_tensor)
            action_probs = torch.softmax(action_scores, dim=0)
            
            log_prob = torch.log(action_probs[action_taken] + 1e-8) # 1e-8 prevents math errors
            total_loss -= log_prob * total_reward 
            
    total_loss.backward()  # Calculate how to change the network
    optimizer.step()       # Actually change the network

# 5. The Evaluator
def compare_to_exact_solution(network, exact_solution_table, grid_width, grid_height):
    correct_moves = 0
    total_squares = grid_width * grid_height
    
    for x in range(grid_width):
        for y in range(grid_height):
            state = [x, y]
            
            state_tensor = torch.FloatTensor(state)
            action_scores = network(state_tensor)
            network_action = torch.argmax(action_scores).item() 
            
            perfect_action = exact_solution_table[x][y]
            
            if network_action == perfect_action:
                correct_moves += 1
                
    accuracy = (correct_moves / total_squares) * 100
    print(f"Network is {accuracy:.2f}% identical to the perfect mathematical solution!")

ModuleNotFoundError: No module named 'torch'

In [4]:
# 1. Create the Environment (5x5 grid)
grid_size = 5
env = GridCarEnv(width=grid_size, height=grid_size)

# Define sensor count k (odd integer between 1 and 9).
k = 7  # choose from {1, 3, 5, 7, 9}
input_size = 4 + k  # network input is 4 vehicle state features + k sensors

# 2. Create the Network (Input is 4+k: [x, y, speed, angle] + k sensors; Output is 4 actions)
net = PolicyNetwork(input_size=input_size, hidden_size=64, num_actions=4)

# 3. Create the Optimizer
optimizer = optim.Adam(net.parameters(), lr=0.01)

# 4. Create a dummy Exact Solution Table
exact_solution_table = []
for x in range(grid_size):
    row = []
    for y in range(grid_size):
        if y == grid_size - 1: row.append(1)
        else: row.append(0)
    exact_solution_table.append(row)

NameError: name 'GridCarEnv' is not defined

In [ ]:
# --- Placeholder observation smoke test ---
# Build a fake observation of length 4 + k: [x, y, speed, angle] + k sensor readings
fake_obs = [0.1, 0.1, 0.01, 0.0] + [0.0] * k
print("Running smoke test: observation length=", len(fake_obs))
obs_tensor = torch.FloatTensor(fake_obs)
with torch.no_grad():
    scores = net(obs_tensor)
    probs = torch.softmax(scores, dim=0)
    action_det = torch.argmax(scores).item()
print("scores shape=", scores.shape, "deterministic action=", action_det)